# AI APIs in the cloud: keys and endpoints

A model served by a provider is called with an HTTP request, authenticated by an API key. This lab does what the "Off-the-Shelf APIs", "Endpoints" and "API Keys" slides show, with no billing account:

1. Read a key from Colab's or Codespaces' secrets
2. Call an API with raw HTTP
3. Make the same call through its SDK
4. Handle errors, quotas and timeouts
5. Revoke and replace a key

The main API is Gemini's (Google): its free tier needs no credit card, it is open in France and the EU, and in the European Economic Area, Switzerland and the UK, Google applies its paid tier's data-use terms to the free quota (your requests aren't used to improve its products; [terms](https://ai.google.dev/gemini-api/terms), October 2026).

The notebook runs without a GPU, on Colab as in the Codespace. Without a key, every cell that calls an API says so and explains what to do: "Run all" still goes through.

## Setup

**A Gemini key**: on [Google AI Studio](https://aistudio.google.com/apikey), sign in with a Google account, then *Create API key*. Copy it, without pasting it into this notebook.

**Store it in the secrets**, under the name `GEMINI_API_KEY`:

- Colab: the key icon (*Secrets*) in the left bar, *Add new secret*, the name and the value, then turn on *Notebook access*.
- Codespaces: on GitHub, *Settings > Codespaces > Codespaces secrets > New secret*, the name and the value, the `shuuchuu/labs-codespace` repository in *Repository access*, then stop and restart the Codespace: a secret only reaches a Codespace (re)started after it, as an environment variable.

## 1. Read a key from the secrets

Write `get_secret(name)`, which returns the value of the `name` secret, or `None` if there is none:

- in Colab, with `userdata.get` from the `google.colab` module, returning `None` when the secret doesn't exist, the notebook has no access to it or the request doesn't go through;
- elsewhere (the Codespace, your machine), where `google.colab` doesn't exist, from the environment variables.

Never print a key's value: a cell's output is saved with the notebook.

**Useful documentation**

- https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py
- https://docs.python.org/3/library/os.html#os.environ
- https://docs.github.com/en/codespaces/managing-your-codespaces/managing-your-account-specific-secrets-for-github-codespaces

In [ ]:
def get_secret(name):
  """The value of the `name` secret, or None if it doesn't exist."""
  # Your code here
  ...

### Additional hints

- `from google.colab import userdata` raises `ImportError` outside Colab: that's where the fallback goes, in a `try:` / `except ImportError:`
- `os.environ.get(name)`: the environment variable, or `None` if it doesn't exist
- `userdata.get(name)` raises `userdata.SecretNotFoundError` (no secret by that name), `userdata.NotebookAccessError` (notebook access not turned on) or `userdata.TimeoutException` (no answer from Colab's interface within 10 seconds)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import os


def get_secret(name):
  """The value of the `name` secret, or None if it doesn't exist."""
  try:
    from google.colab import userdata
  except ImportError:  # outside Colab (Codespaces): an environment variable
    return ...
  try:
    return userdata.get(name)
  except (userdata.SecretNotFoundError, ...):
    return None
```

### Solution

In [ ]:
import os


def get_secret(name):
  """The value of the `name` secret, or None if it doesn't exist."""
  try:
    from google.colab import userdata
  except ImportError:  # outside Colab (Codespaces): an environment variable
    return os.environ.get(name)
  try:
    return userdata.get(name)
  except (
    userdata.SecretNotFoundError,
    userdata.NotebookAccessError,
    userdata.TimeoutException,
  ):
    return None

## 2. Call the API with raw HTTP

The Gemini API has an endpoint compatible with OpenAI's: the request of the "Endpoints" slide works there as is, with the URL, the model and the key below. The model is a pinned version (`gemini-3.5-flash-lite`, stable, free in the free tier), not an alias like `gemini-flash-latest`, which Google points to each new version.

In [ ]:
import httpx

GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"
NO_GEMINI_KEY = (
  "No GEMINI_API_KEY key: create it and store it in the secrets (see"
  ' "Setup"), then rerun the cell that reads it.'
)

GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
print("GEMINI_API_KEY:", "found" if GEMINI_API_KEY else "missing")

If the key is there, send a question of your choice to the `MODEL` model with `httpx.post`, the key in the `Authorization` header, with a timeout, and raise an exception if the status code is an error. Print the answer's text, then its `usage` field: the tokens counted, what the provider bills (and what the quota counts). Without a key, print `NO_GEMINI_KEY`.

**Useful documentation**

- https://www.python-httpx.org/api/#helper-functions
- https://www.python-httpx.org/api/#response
- https://ai.google.dev/gemini-api/docs/openai

In [ ]:
# Your code here

### Additional hints

- `httpx.post(url, headers=..., json=..., timeout=60)`: `json=` encodes the dictionary and sets the `Content-Type: application/json` header; without `timeout`, httpx gives up after 5 seconds, too little for an LLM
- the header: `{"Authorization": f"Bearer {GEMINI_API_KEY}"}`
- the body: `{"model": MODEL, "messages": [{"role": "user", "content": "..."}]}`
- `response.raise_for_status()` raises `httpx.HTTPStatusError` on a 4xx or 5xx code
- `response.json()["choices"][0]["message"]["content"]`: the text; `response.json()["usage"]`: the tokens of the question (`prompt_tokens`) and of the answer (`completion_tokens`)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
if GEMINI_API_KEY:
  response = httpx.post(
    GEMINI_URL,
    headers={"Authorization": ...},
    json={
      "model": MODEL,
      "messages": [{"role": "user", "content": ...}],
    },
    timeout=60,
  )
  response.raise_for_status()
  body = response.json()
  print(body["choices"][0]["message"]["content"])
  print(...)
else:
  print(NO_GEMINI_KEY)
```

### Solution

In [ ]:
if GEMINI_API_KEY:
  response = httpx.post(
    GEMINI_URL,
    headers={"Authorization": f"Bearer {GEMINI_API_KEY}"},
    json={
      "model": MODEL,
      "messages": [
        {"role": "user", "content": "In two sentences: what is an API key?"}
      ],
    },
    timeout=60,
  )
  response.raise_for_status()
  body = response.json()
  print(body["choices"][0]["message"]["content"])
  print(body["usage"])
else:
  print(NO_GEMINI_KEY)

## 3. The same call through the SDK

Ask the same question again with Google's SDK, `google-genai`: a client created with the key and a timeout, then `generate_content`. Print the answer's text and the tokens counted. Compare with part 2: what does the SDK do for you, and what does it hide from you?

**Useful documentation**

- https://ai.google.dev/gemini-api/docs/text-generation
- https://googleapis.github.io/python-genai/

In [ ]:
# Your code here

### Additional hints

- `genai.Client(api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=60_000))`: the SDK's timeout is in **milliseconds**; without `api_key`, the client looks for the `GEMINI_API_KEY` or `GOOGLE_API_KEY` environment variables (not Colab's secrets, which aren't any)
- `client.models.generate_content(model=MODEL, contents="...")` returns a typed response: `response.text`, `response.usage_metadata`
- the SDK picks the URL, sets the key's header (`x-goog-api-key`, the native Gemini API's), encodes and decodes the JSON, and turns an HTTP error into a `google.genai.errors.APIError` exception

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from google import genai
from google.genai import types

if GEMINI_API_KEY:
  client = genai.Client(
    api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=...)
  )
  response = client.models.generate_content(model=MODEL, contents=...)
  print(response.text)
  print(...)
else:
  print(NO_GEMINI_KEY)
```

### Solution

In [ ]:
from google import genai
from google.genai import types

if GEMINI_API_KEY:
  client = genai.Client(
    api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=60_000)
  )
  response = client.models.generate_content(
    model=MODEL, contents="In two sentences: what is an API key?"
  )
  print(response.text)
  print(response.usage_metadata)
else:
  print(NO_GEMINI_KEY)

## 4. Errors, quotas and timeouts

An API fails, and the code calling it must know when to retry:

| Code | Cause | What to do |
|---|---|---|
| 400, 401, 403 | Malformed request, missing, invalid or revoked key, insufficient rights | Don't retry: fix the request or the key |
| 429 | Quota or rate limit reached (requests per minute, per day…) | Wait (the `Retry-After` header, when present, says how many seconds), then retry |
| 500, 502, 503, 504 | The provider is down or overloaded | Retry, waiting longer and longer |
| Timeout | Slow network, overloaded server | Retry, a few times at most |

The Gemini free tier's limits depend on the model and the project: Google AI Studio shows them ([rate limits](https://ai.google.dev/gemini-api/docs/rate-limits)).

### 4.1 Retry when it makes sense

Write `post_json(client, url, payload, max_attempts=4, base_delay=1.0)`, which sends `payload` as JSON with the httpx client `client` (which already carries the key and the timeout) and returns the successful response:

- after a 429, a 5xx or a timeout, it waits then retries, `max_attempts` times in all at most;
- the wait is `Retry-After` when the response gives it, else `base_delay`, then twice as long at each new attempt (exponential backoff);
- any other error, and the last attempt's error, raise their exception (`httpx.HTTPStatusError`, `httpx.TimeoutException`).

Print each new attempt and its wait. Part 4.2's cells try your function.

**Useful documentation**

- https://www.python-httpx.org/api/#client
- https://www.python-httpx.org/exceptions/
- https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Retry-After

In [ ]:
def post_json(client, url, payload, max_attempts=4, base_delay=1.0):
  """POST `payload` as JSON, retrying after a 429, a 5xx or a timeout.

  Returns the successful response; raises httpx.HTTPStatusError for another error
  (request or key refused) and for the last attempt's error.
  """
  # Your code here
  ...

#### Additional hints

- `client.post(url, json=payload)` raises `httpx.TimeoutException` when the timeout is exceeded
- `response.status_code`: the code; `response.raise_for_status()` raises `httpx.HTTPStatusError` on an error, and returns the response otherwise
- `response.headers.get("Retry-After")`: `None` when the header isn't there, a string otherwise (`"1"`)
- the wait of attempt `n` (from 1): `base_delay * 2 ** (n - 1)`, that is 1, 2, 4 seconds; `time.sleep(delay)`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
import time

RETRY_STATUS = {429, 500, 502, 503, 504}


def post_json(client, url, payload, max_attempts=4, base_delay=1.0):
  """POST `payload` as JSON, retrying after a 429, a 5xx or a timeout."""
  for attempt in range(1, max_attempts + 1):
    try:
      response = client.post(url, json=payload)
    except httpx.TimeoutException:
      if attempt == max_attempts:
        raise
      delay = ...
      print(f"  attempt {attempt}: timeout, retrying in {delay:.1f} s")
    else:
      if response.status_code not in RETRY_STATUS or attempt == max_attempts:
        return response.raise_for_status()
      delay = float(response.headers.get("Retry-After", ...))
      print(f"  attempt {attempt}: {response.status_code}, retrying in {delay:.1f} s")
    time.sleep(delay)
```

#### Solution

In [ ]:
import time

RETRY_STATUS = {429, 500, 502, 503, 504}


def post_json(client, url, payload, max_attempts=4, base_delay=1.0):
  """POST `payload` as JSON, retrying after a 429, a 5xx or a timeout.

  Returns the successful response; raises httpx.HTTPStatusError for another error
  (request or key refused) and for the last attempt's error.
  """
  for attempt in range(1, max_attempts + 1):
    try:
      response = client.post(url, json=payload)
    except httpx.TimeoutException:
      if attempt == max_attempts:
        raise
      delay = base_delay * 2 ** (attempt - 1)
      print(f"  attempt {attempt}: timeout, retrying in {delay:.1f} s")
    else:
      if response.status_code not in RETRY_STATUS or attempt == max_attempts:
        return response.raise_for_status()
      delay = float(
        response.headers.get("Retry-After", base_delay * 2 ** (attempt - 1))
      )
      print(f"  attempt {attempt}: {response.status_code}, retrying in {delay:.1f} s")
    time.sleep(delay)

### 4.2 Try it on a fake server, then on the real one

Causing a 429 on the real API would use up your quota: `httpx.MockTransport` replaces the network with a function, which replays a series of responses here (an HTTP code, or `"timeout"` for a timeout). For each scenario, how many requests does your function send, and how long does it wait?

In [ ]:
import time


def fake_api(replies):
  """An httpx client whose "server" replays `replies`, one per request."""
  calls = []

  def handler(request):
    reply = replies[len(calls)]
    calls.append(reply)
    if reply == "timeout":
      raise httpx.ReadTimeout("timeout (simulated)", request=request)
    if reply == 200:
      return httpx.Response(200, json={"choices": [{"message": {"content": "Hello"}}]})
    headers = {"Retry-After": "1"} if reply == 429 else {}
    return httpx.Response(reply, headers=headers, json={"error": f"error {reply}"})

  return httpx.Client(transport=httpx.MockTransport(handler)), calls


scenarios = {
  "quota reached twice": [429, 429, 200],
  "one timeout": ["timeout", 200],
  "transient outage": [503, 503, 200],
  "invalid key": [401],
  "quota used up": [429, 429, 429, 429],
}
for name, replies in scenarios.items():
  print(f"{name}: {replies}")
  client, calls = fake_api(replies)
  start = time.perf_counter()
  try:
    response = post_json(client, "https://api.example/v1/chat", {}, base_delay=0.5)
    outcome = None if response is None else response.status_code
  except httpx.HTTPStatusError as error:
    outcome = f"gave up, {error.response.status_code}"
  except httpx.TimeoutException:
    outcome = "gave up, timeout"
  elapsed = time.perf_counter() - start
  print(f"  → {outcome}, {len(calls)} request(s) in {elapsed:.1f} s")

The real API, with an invalid key (that takes no account). Gemini answers `400 INVALID_ARGUMENT` (reason `API_KEY_INVALID`), not `401` like most OpenAI-compatible APIs: read the error's body, not only its code. The SDK raises `google.genai.errors.ClientError` (4xx) or `ServerError` (5xx), both `APIError`s; it can retry by itself (`types.HttpOptions(retry_options=types.HttpRetryOptions(attempts=4))`).

In [ ]:
from google import genai
from google.genai import errors

question = {"model": MODEL, "messages": [{"role": "user", "content": "Hello"}]}
with httpx.Client(headers={"Authorization": "Bearer invalid-key"}, timeout=30) as http:
  try:
    post_json(http, GEMINI_URL, question)
  except httpx.HTTPStatusError as error:
    print(error.response.status_code, error.response.text[:200])
  except httpx.TransportError as error:
    print("No network:", error)

bad_client = genai.Client(api_key="invalid-key")
try:
  bad_client.models.generate_content(model=MODEL, contents="Hello")
except errors.APIError as error:
  print(type(error).__name__, error.code, error.status, error.message)
except httpx.TransportError as error:
  print("No network:", error)

## 5. Revoke and replace a key

Suppose `GEMINI_API_KEY` has leaked (pasted in a cell, pushed to GitHub). The next cell keeps its current value: run it **before** touching the key.

In [ ]:
old_key = GEMINI_API_KEY

Write `key_is_valid(key)`, which returns `True` if Gemini accepts the key, `False` otherwise (printing the error's code). Try it on a made-up key, then rotate the key:

1. on [Google AI Studio's keys page](https://aistudio.google.com/apikey), create a new key, then delete the old one: it no longer works anywhere;
2. store the new value in the `GEMINI_API_KEY` secret (Codespaces: then restart the Codespace, and rerun the notebook down to the previous cell);
3. read the secret again, and check the old and the new key.

**Useful documentation**

- https://ai.google.dev/gemini-api/docs/api-key
- https://googleapis.github.io/python-genai/genai.html#genai.models.Models.get

In [ ]:
def key_is_valid(key):
  """True if Gemini accepts `key`."""
  # Your code here
  ...

### Additional hints

- `client.models.get(model=MODEL)` reads the model's card, consuming no tokens; with a refused key, it raises `google.genai.errors.ClientError`, whose `error.code` is 400
- keep the client in a variable: a `genai.Client(...)` that no variable holds is closed before it sends its request
- a made-up key, to try: `"madeup-key"`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
from google import genai
from google.genai import errors


def key_is_valid(key):
  """True if Gemini accepts `key`."""
  client = genai.Client(api_key=...)
  try:
    client.models.get(model=MODEL)
  except errors.ClientError as error:
    print("  refused:", error.code)
    return False
  return True


print("made-up key:", key_is_valid("madeup-key"))
if old_key:
  GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
  print("old key:", key_is_valid(...))
  print("new key:", key_is_valid(...))
```

### Solution

In [ ]:
from google import genai
from google.genai import errors


def key_is_valid(key):
  """True if Gemini accepts `key`."""
  client = genai.Client(api_key=key)
  try:
    client.models.get(model=MODEL)
  except errors.ClientError as error:
    print("  refused:", error.code)
    return False
  return True


print("made-up key:", key_is_valid("madeup-key"))
if old_key:
  GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
  print("old key:", key_is_valid(old_key))
  print("new key:", key_is_valid(GEMINI_API_KEY))
else:
  print(NO_GEMINI_KEY)

## 6. Clean up

Delete the keys you no longer need, in Google AI Studio. One key per use, and not one more.